# The Determinants of Turnout

In [ ]:
import stata_setup

stata_setup.config('/Applications/StataNow', 'mp')

In [ ]:
%%stata

clear all
set more off
set varabbrev off
version 18

* Paths
global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-governance/processed_data"
global TABLES "tables"


. 
. clear all

. set more off

. set varabbrev off

. version 18

. 
. * Paths
. global PROCESSED_DATA "/Users/yichenluo/Library/CloudStorage/Dropbox/dao-gove
> rnance/processed_data"

. global TABLES "tables"

. 


In [ ]:
%%stata

import delimited using "$PROCESSED_DATA/proposals_panel.csv", clear

* Parse date
gen day = date(date, "YMD")
format day %td
gen year = year(day)

* Independent variables
local complexity multi_choices weighted quorum delegation
local topic protocol_security treasury_expenditure user_incentive_increase tokenomics voting_mechanism_change


* Label variables
label var weighted "\${\it Weighted}_{i,t}\$"
label var quorum "\${\it Quorum}_{i,t}\$"
label var multi_choices "\${\it Multiple Choices}_{i,t}\$"
label var have_discussion "\${\it Discussion}_{i,t}\$"
label var delegation "\${\it Delegation}_{i,t}\$"
label var non_whale_user      "\${\it User}_{i,t}^{\it Small}\$"
label var concensus_diff "\${\it \Delta Concensus}_{i,t}\$"

## Proposal Characteristics

In [ ]:
%%stata

******************************************************
* PANEL REGRESSIONS
******************************************************

eststo clear

    * VIF test
    qui reg non_whale_participation `complexity' `topic'
    estat vif

    * Run baseline regression
    reghdfe non_whale_participation `complexity' have_discussion , absorb(year categories) 
    eststo baseline
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"


    * Run regression with discussion characteristics    
    reghdfe non_whale_participation `complexity' concensus_diff, absorb(year categories) 
    eststo discussion
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"


    * Run regression with user characteristics
    reghdfe non_whale_participation `complexity' have_discussion non_whale_user, absorb(year categories)
    eststo user
    estadd local fe_proposal  "Y"
    estadd local fe_time  "Y"

* Export LaTeX table
esttab                                                           ///
    baseline discussion user                                     ///
    using "$TABLES/reg_participation_char.tex", replace          ///
    se star(* 0.10 ** 0.05 *** 0.01) label nogaps nocon nonumbers ///
    nonotes booktabs nomtitles                                   ///
    b(%9.3f) se(%9.2f)                                           ///
    order(delegation have_discussion concensus_diff non_whale_user multi_choices weighted quorum) ///
    mgroups("\${\it Participation}^{Small}_{i,t}\$",             ///
            span                                                 ///
            pattern(1 0 0)                                       ///
            prefix(\multicolumn{@span}{c}{)                      ///
            suffix(})                                            ///
            erepeat(\cmidrule(lr){@span}) )                      ///
    posthead(\cmidrule(lr){2-4} & \multicolumn{1}{c}{Full} & \multicolumn{1}{c}{Discussion \$\geq\$ 2} & \multicolumn{1}{c}{Have Dapp} \\ \cmidrule(lr){2-4} & \multicolumn{1}{c}{(1)} & \multicolumn{1}{c}{(2)} & \multicolumn{1}{c}{(3)} \\ \midrule) ///
            substitute("\_" "_")                                ///
    stats(fe_proposal fe_time N r2_a,                            ///
        fmt(0 0 %9.0fc %9.3f)                                    ///
         labels("Category FE" "Year FE" "Observations" "Adjusted R²"))